<a href="https://colab.research.google.com/github/laoneya31/IAT360-Group-18/blob/main/notebooks/Lavanya_Licences_Freeze_FinalEvaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><h1><b>Licences, Frozen Layers and Final Evaluation</b></h1></center>
<center>IAT 360 Group 18 (Lavanya)</center>

**Tasks:** licence check (record it in `docs/DATA_SOURCES.md`), tune `freeze`, final evaluation on our 120-photo test set, pick worked and failed examples.

## 1. Environment

Use a GPU runtime: **Runtime > Change runtime type > T4 GPU**. The shared Drive folder must be in your My Drive: right-click `IAT360-Group 18` in *Shared with me* and choose **Organize > Add shortcut**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Clone our repo and the official YOLO repo. YOLO is pinned to one release so all four members train with identical code.

In [ ]:
import os, sys
REPO = "/content/IAT360-Group-18"
YOLO_TAG = "v8.4.204"

if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone https://github.com/laoneya31/IAT360-Group-18 {REPO}
# The folder must not be called "ultralytics": /content is on sys.path, so it would hide the real package.
if not os.path.exists("/content/yolo_repo"):
    !git -c advice.detachedHead=false clone -q --depth 1 --branch {YOLO_TAG} https://github.com/ultralytics/ultralytics /content/yolo_repo
%pip install -q /content/yolo_repo kagglehub
%cd {REPO}

In [ ]:
import ultralytics
ultralytics.checks()
from ultralytics import YOLO

sys.path.insert(0, f"{REPO}/src")
import fruit_yolo as fy
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import display

fy.check_config()
print("Team Drive folder:", fy.find_team_drive())
print("Baseline settings:", fy.BASELINE)

## Hyperparameter tuning

`freeze`, baseline none. Freezing the first N layers keeps the pretrained features and trains faster, but gives the model less room to learn rot.

Only this one value changes. Everything else stays at the shared baseline, so the difference from the baseline can be attributed to it.

In [ ]:
assert fy.fetch_dataset(), "Dataset is not on Drive yet."
OWNER = "Lavanya"
PARAM, VALUES = fy.TUNING_PLAN[OWNER]
RUNS = [f"{PARAM}_{v}" for v in VALUES]

for name, value in zip(RUNS, VALUES):
    weights = fy.train_run(name, OWNER, overrides={PARAM: value})
    fy.evaluate(name, weights, imgsz=value if PARAM == "imgsz" else None)

In [ ]:
table = fy.compare_runs(names=["baseline"] + RUNS)
display(table)
fy.plot_comparison(table, f"{PARAM}: mAP@0.5:0.95 on the validation set",
                   save_to=fy.output_root() / "figures" / f"tuning_{PARAM}.png")
plt.show()
for name in RUNS:
    print(name, fy.fit_report(name))

### My results

*TODO: which value won, by how much, and why you think that happened. Copy this into the report.*

## Final evaluation on our 120 photos

Run this once Philip has trained the final model. The test set is used **only here**, so it stays a fair measure.

In [ ]:
assert fy.fetch_dataset(), "Dataset is not on Drive yet."
weights = fy.output_root() / "weights" / "final.pt"
results = {split: fy.evaluate("final", weights, split=split) for split in ("val", "test")}
display(pd.DataFrame({s: {k: r[k] for k in fy.METRIC_COLUMNS} for s, r in results.items()}).round(3))
display(pd.DataFrame(results["test"]["per_class"]).T.round(3))

Confusion matrix on the test set.

In [ ]:
figures = fy.output_root() / "figures"
display(Image.open(figures / "final_test" / "confusion_matrix_normalized.png").resize((720, 540)))

Worked and failed examples. Green boxes are true labels, orange boxes are predictions.

In [ ]:
scored = fy.score_images(weights, split="test")
fy.show_examples(scored, correct=True, n=6, save_to=figures / "final_test" / "examples_correct.png"); plt.show()
fy.show_examples(scored, correct=False, n=6, save_to=figures / "final_test" / "examples_incorrect.png"); plt.show()

### Results

*TODO: how much worse is the model on our store photos than on the public validation set, and which scenes fail (bagged fruit, early rot, lighting)?*